# Turkish RAG Benchmark - Retrieval Evaluation (Standalone Version)
Bu Notebook, Google Colab üzerinde RAG sistemlerinin performansını ölçmek için tasarlanmıştır. Hiçbir dış Python dosyasına (src/ klasörüne) ihtiyaç duymadan **tek başına** çalışır.

**ÖNEMLİ:** Çalıştırmadan önce Colab menüsünden `Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU` seçili olduğundan emin olun.


In [ ]:
# 1. Google Drive'ı Bağla ve Proje Dizinine Git
from google.colab import drive
import os

drive.mount('/content/drive')

# NOT: Projenizi Drive'da nereye yüklediyseniz aşağıdaki yolu güncelleyin.
PROJECT_PATH = '/content/drive/MyDrive/turkish-rag-benchmark'
os.chdir(PROJECT_PATH)
print("Current Directory:", os.getcwd())


In [ ]:
# 2. Gerekli Kütüphaneleri Kur
!pip install -U langchain==0.2.14 langchain-community==0.2.12 langchain-core==0.2.35 langchain-huggingface==0.0.3 chromadb sentence-transformers rank_bm25 pandas tqdm langchain-google-genai
print("\nKURULUM TAMAMLANDI! Lütfen Colab menüsünden 'Runtime -> Restart session' yapın ve 3. hücreden devam edin.")


In [ ]:
# 3. Ortam Değişkenleri
import os
import getpass
import string
import json
import glob
from typing import List
import pandas as pd

os.environ['COLAB_GPU'] = '1'

# Sadece Grup 5 (HyDE) için Gemini API Key gereklidir (boş geçebilirsiniz)
if 'GEMINI_API_KEY' not in os.environ:
    try:
        os.environ['GEMINI_API_KEY'] = getpass.getpass('Enter Gemini API Key (optional unless running Group 5): ')
    except:
        pass


In [ ]:
# 4. Modül: Data Loader (Veri Yükleyici)
from langchain_core.documents import Document

def load_markdown_documents(base_dir: str):
    documents = []
    search_pattern = os.path.join(base_dir, "**", "*.md")
    file_paths = glob.glob(search_pattern, recursive=True)
    for file_path in file_paths:
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()
            file_name = os.path.basename(file_path)
            doc = Document(page_content=content, metadata={"source": file_name, "file_path": file_path})
            documents.append(doc)
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
    return documents

def load_qa_dataset(benchmark_dir: str):
    combined_dataset = []
    search_pattern = os.path.join(benchmark_dir, "*.json")
    json_files = glob.glob(search_pattern)
    for file_path in json_files:
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                if isinstance(data, list):
                    combined_dataset.extend(data)
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
    return combined_dataset


In [ ]:
# 5. Modül: RAG Builder (LangChain Yapıcıları)
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers.ensemble import EnsembleRetriever
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun

def get_text_splitter(strategy="recursive_512"):
    if strategy == "recursive_512": return RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=128)
    elif strategy == "recursive_256": return RecursiveCharacterTextSplitter(chunk_size=256, chunk_overlap=50)
    elif strategy == "recursive_1024": return RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=200)
    elif strategy == "markdown": return MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "Header 1"), ("##", "Header 2"), ("###", "Header 3")])
    raise ValueError(f"Unknown strategy: {strategy}")

def split_documents(docs, strategy="recursive_512"):
    splitter = get_text_splitter(strategy)
    if strategy == "markdown":
        split_docs = []
        for doc in docs:
            md_splits = splitter.split_text(doc.page_content)
            for split in md_splits:
                split_docs.append(Document(page_content=split.page_content, metadata={**doc.metadata, **split.metadata}))
        return split_docs
    return splitter.split_documents(docs)

def get_embedding_model(model_name="BAAI/bge-m3"):
    return HuggingFaceEmbeddings(model_name=model_name, model_kwargs={'device': 'cuda'}, encode_kwargs={'normalize_embeddings': True})

def build_vector_store(docs, embedding_model_name, chunk_strategy, persist_dir):
    embeddings = get_embedding_model(embedding_model_name)
    collection_name = f"{embedding_model_name.replace('/', '_')}_{chunk_strategy}".lower()
    if os.path.exists(persist_dir) and os.listdir(persist_dir):
        return Chroma(collection_name=collection_name, embedding_function=embeddings, persist_directory=persist_dir)
    vectordb = Chroma(collection_name=collection_name, embedding_function=embeddings, persist_directory=persist_dir)
    batch_size = 5000
    for i in range(0, len(docs), batch_size):
        vectordb.add_documents(docs[i:i + batch_size])
    return vectordb

def get_bm25_retriever(docs, k=5):
    bm25 = BM25Retriever.from_documents(docs)
    bm25.k = k
    return bm25

def get_hybrid_retriever(dense, bm25, weights=[0.5, 0.5]):
    return EnsembleRetriever(retrievers=[dense, bm25], weights=weights)

def get_reranker_retriever(base_retriever, model_name="BAAI/bge-reranker-v2-m3", top_n=5):
    compressor = CrossEncoderReranker(model=HuggingFaceCrossEncoder(model_name=model_name, model_kwargs={'device': 'cuda'}), top_n=top_n)
    return ContextualCompressionRetriever(base_compressor=compressor, base_retriever=base_retriever)

class HyDERetriever(BaseRetriever):
    base_retriever: BaseRetriever
    llm: ChatGoogleGenerativeAI
    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun) -> List[Document]:
        ans = self.llm.invoke(f"Lütfen aşağıdaki soruya cevap verecek örnek bir akademik paragraf yazın:\nSoru: {query}").content
        return self.base_retriever.invoke(f"{query}\n\n{ans}")



In [ ]:
# 6. Modül: Retrieval Evaluator (Değerlendirici)
def normalize_text(s: str) -> str:
    s = s.lower().strip()
    return s.translate(str.maketrans('', '', string.punctuation))

def is_hit(ground_truth: str, retrieved_content: str) -> bool:
    gt_norm = normalize_text(ground_truth)
    ret_norm = normalize_text(retrieved_content)
    if gt_norm in ret_norm:
        return True
    # Jaccard
    a, b = set(gt_norm.split()), set(ret_norm.split())
    if not a or not b: return False
    c = a.intersection(b)
    if (float(len(c)) / (len(a) + len(b) - len(c))) > 0.3: return True
    return False

def evaluate_dataset(retriever, dataset: List[dict]):
    from tqdm.auto import tqdm
    total_hits, total_rr = 0, 0.0
    for item in tqdm(dataset, desc="Evaluating"):
        try:
            retrieved_docs = retriever.invoke(item['question'])
            for idx, doc in enumerate(retrieved_docs):
                if is_hit(item['ground_truth'], doc.page_content):
                    total_hits += 1
                    total_rr += 1.0 / (idx + 1)
                    break
        except Exception as e:
            print(f"Error: {e}")
            
    t = len(dataset)
    return {"Hit Rate": total_hits/t if t>0 else 0, "MRR": total_rr/t if t>0 else 0, "Total": t}

results = {}
def print_and_save(group_name, sys_name, res):
    print(f"\n[{group_name}] {sys_name} -> Hit Rate: {res['Hit Rate']:.4f} | MRR: {res['MRR']:.4f}")
    results[sys_name] = res


In [ ]:
# 7. Veri Yükleme İşlemi
print("Dokümanlar yükleniyor...")
md_dir = os.path.join("data", "processed", "stage2_cleaned")
docs = load_markdown_documents(md_dir)

print("Soru veri seti yükleniyor...")
benchmark_dir = os.path.join("data", "benchmark")
qa_data = load_qa_dataset(benchmark_dir)

# Test Boyutu Ayarı: Sadece 100 soruyla hızlı test yapmak için:
SAMPLE_SIZE = 100 
import random
random.seed(42)
qa_sample = random.sample(qa_data, SAMPLE_SIZE) if SAMPLE_SIZE < len(qa_data) else qa_data
print(f"Toplam {len(docs)} doküman, test edilecek soru sayısı: {len(qa_sample)}")


## GRUP 1: Embedding Karşılaştırması
Sabit Chunk (512/128) ve Dense Retrieval


In [ ]:
docs_512 = split_documents(docs, "recursive_512")
db_path_base = "results/chroma_db"
os.makedirs(db_path_base, exist_ok=True)

for i, emb_name in enumerate(["BAAI/bge-m3", "intfloat/multilingual-e5-large", "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"], 1):
    print(f"\n--- RAG-{i} Indexing: {emb_name} ---")
    retriever = build_vector_store(docs_512, emb_name, "recursive_512", os.path.join(db_path_base, f"g1_{emb_name.replace('/', '_')}")).as_retriever(search_kwargs={"k": 5})
    print_and_save("GRUP 1", f"RAG-{i} ({emb_name})", evaluate_dataset(retriever, qa_sample))


## Ara Karar


In [ ]:
BEST_EMBEDDING = "BAAI/bge-m3" # Tabloya göre güncelleyin


## GRUP 2: Chunk Stratejisi


In [ ]:
for i, strat in enumerate(["recursive_256", "recursive_1024", "markdown"], 4):
    print(f"\n--- RAG-{i} Indexing: {strat} ---")
    retriever = build_vector_store(split_documents(docs, strat), BEST_EMBEDDING, strat, os.path.join(db_path_base, f"g2_{strat}")).as_retriever(search_kwargs={"k": 5})
    print_and_save("GRUP 2", f"RAG-{i} ({strat})", evaluate_dataset(retriever, qa_sample))


## Ara Karar


In [ ]:
BEST_CHUNK = "recursive_512" # Önceki tüm testler içinde en iyisi


## GRUP 3: Retrieval Stratejisi
BM25 ve Hybrid Sistemler


In [ ]:
best_docs = split_documents(docs, BEST_CHUNK)
best_vectorstore = build_vector_store(best_docs, BEST_EMBEDDING, BEST_CHUNK, os.path.join(db_path_base, f"best_{BEST_CHUNK}"))

dense_retriever = best_vectorstore.as_retriever(search_kwargs={"k": 5})
bm25_retriever = get_bm25_retriever(best_docs, k=5)

print_and_save("GRUP 3", "RAG-7 (BM25)", evaluate_dataset(bm25_retriever, qa_sample))

hybrid_retriever = get_hybrid_retriever(dense_retriever, bm25_retriever)
print_and_save("GRUP 3", "RAG-8 (Hybrid)", evaluate_dataset(hybrid_retriever, qa_sample))


## GRUP 4: Reranker


In [ ]:
hybrid_for_rerank = get_hybrid_retriever(best_vectorstore.as_retriever(search_kwargs={"k": 10}), get_bm25_retriever(best_docs, k=10))
reranker_retriever = get_reranker_retriever(hybrid_for_rerank, top_n=5)
print_and_save("GRUP 4", "RAG-9 (Hybrid+Reranker)", evaluate_dataset(reranker_retriever, qa_sample))


## GRUP 5: Gelişmiş Mimariler


In [ ]:
try:
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)
    print_and_save("GRUP 5", "RAG-10 (HyDE+Dense)", evaluate_dataset(HyDERetriever(base_retriever=dense_retriever, llm=llm), qa_sample))
    print_and_save("GRUP 5", "RAG-11 (MultiQuery+Hybrid+Reranker)", evaluate_dataset(get_reranker_retriever(MultiQueryRetriever.from_llm(retriever=hybrid_for_rerank, llm=llm), top_n=5), qa_sample))
except Exception as e:
    print(f"Grup 5 hatası: {e}")


## SONUÇLAR


In [ ]:
df = pd.DataFrame.from_dict(results, orient='index').sort_values(by='Hit Rate', ascending=False)
display(df)
df.to_csv("results/comparison.csv")
